# Group 1 Baseline Workflow (Storyteller Notebook)

This notebook is the **clean orchestration layer** for Group 1 baseline.
All core logic lives in `src/`; notebook cells call those modules stage-by-stage.


## Stage Map

1. Setup and environment check
2. Data acquisition + prep (COCO -> alignment -> instruction format)
3. Tokenization (stage 1 and stage 2)
4. CLIP feature precompute
5. Manifest build
6. Stage 1 training
7. Stage 2 training
8. Artifacts + quick validation


In [ ]:
from pathlib import Path
import json
import os
import sys

NB_DIR = Path.cwd()
PROJECT_ROOT = NB_DIR.parent
CONFIG_PATH = PROJECT_ROOT / "configs" / "workflow_paths.json"
DOTENV_PATH = PROJECT_ROOT / ".env"

# Add project root once; import from src.* modules below.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config_loader import load_dotenv_file, load_json_config

load_dotenv_file(DOTENV_PATH)
print("PROJECT_ROOT =", PROJECT_ROOT)
print("CONFIG_PATH =", CONFIG_PATH)
print("HF_TOKEN loaded =", bool(os.environ.get("HF_TOKEN")))


In [ ]:
# Load central path config from configs/workflow_paths.json
cfg = load_json_config(CONFIG_PATH, PROJECT_ROOT)

Path(cfg["artifacts_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(cfg, indent=2))


## Stage 0: Environment Check

Run this before heavy steps to confirm required imports are available.


### Checklist - Before Stage 0
- [ ] `.venv` is active and selected as notebook kernel
- [ ] `python scripts/check_env.py --profile all` passes in terminal
- [ ] `.env` exists and contains `HF_TOKEN`
- [ ] Config file exists: `configs/workflow_paths.json`


In [ ]:
# Run from `code/group1_baseline` root in terminal if this cell fails:
# python scripts/check_env.py --profile core
# python scripts/check_env.py --profile notebook
# python scripts/check_env.py --profile tpu


## Stage 1: Data Acquisition + Prep (COCO -> Alignment -> Instruction Format)

Inputs:
- COCO data (downloaded if missing)

Outputs:
- `cfg["stage1_alignment_json"]`
- `cfg["stage1_chat_json"]`


### Checklist - Before Stage 1
- [ ] Enough disk space for COCO images + annotations
- [ ] `data/raw/` path is correct in config
- [ ] You understand overwrite policy (`overwrite=False` by default)
- [ ] If regenerating, delete old Stage 1 processed outputs first


In [ ]:
from src.data_prep.stage1_pipeline import run_stage1_data_prep

coco_path = Path(cfg["coco_json"])
alignment_path = Path(cfg["stage1_alignment_json"])
chat_path = Path(cfg["stage1_chat_json"])

alignment, chat_rows, mode, status = run_stage1_data_prep(
    project_root=PROJECT_ROOT,
    coco_json_path=coco_path,
    alignment_path=alignment_path,
    chat_path=chat_path,
    seed=42,
    overwrite=False,
    download=True,
    extract=True,
)

for k, pth in status.items():
    print(f"{k}: {pth} -> {'OK' if pth.exists() else 'MISSING'}")

print(f"Stage 1 mode: {mode}")
print("alignment rows:", len(alignment))
print("chat rows:", len(chat_rows))
print("sample:", chat_rows[0] if chat_rows else "<empty>")


### Stage 1 Sanity Visualization

Show a few real images from `chat_rows` with prompt/response snippets.


In [ ]:
import random
from PIL import Image
import matplotlib.pyplot as plt

if not chat_rows:
    raise RuntimeError("chat_rows is empty; run Stage 1 first.")

n = min(4, len(chat_rows))
examples = random.sample(chat_rows, k=n)

fig, axes = plt.subplots(1, n, figsize=(5 * n, 4))
if n == 1:
    axes = [axes]

for ax, sample in zip(axes, examples):
    img_path = Path(cfg["image_root"]) / sample["image"]
    img = Image.open(img_path).convert("RGB")
    ax.imshow(img)
    ax.axis("off")
    title = sample["instruction"][:45]
    ax.set_title(title)
    print("image:", sample["image"])
    print("instruction:", sample["instruction"])
    print("response:", sample["response"][:140], "...")
    print("-")

plt.tight_layout()
plt.show()


## Stage 2: Tokenization (Executable)

This cell runs tokenization directly using `src.training.tokenization`.
- Always builds Stage 1 tokenized data if missing
- Builds Stage 2 tokenized data only when `stage2_input_json` exists


### Stage 2 Note (Group 1 Reproduction)

Group 1 workflow effectively reused Stage 1 chat-format data for Stage 2 tokenization,
but with a longer sequence length (`max_len=256`).

So this notebook follows the same rule:
- if `stage2_input_json` exists, use it
- otherwise fallback to `stage1_chat_json`
- output still goes to `stage2_tokenized_json`


### Checklist - Before Stage 2
- [ ] Stage 1 outputs exist (`alignment_chat.json`)
- [ ] HF login is valid: `hf auth whoami`
- [ ] Access to gated model/tokenizer is approved (if using LLaMA)
- [ ] On TPU VMs, set `HF_HUB_ENABLE_HF_TRANSFER=0` if model loading hangs


In [ ]:
# TPU loader safety (from original notebook note).
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
print("HF_HUB_ENABLE_HF_TRANSFER=", os.environ["HF_HUB_ENABLE_HF_TRANSFER"])


In [ ]:
from pathlib import Path
from src.training.tokenization_pipeline import run_tokenization_pipeline

result = run_tokenization_pipeline(
    tokenizer_id=cfg.get("tokenizer_id", "meta-llama/Llama-3.2-1B-Instruct"),
    stage1_input_json=Path(cfg["stage1_chat_json"]),
    stage1_output_json=Path(cfg["stage1_tokenized_json"]),
    stage2_input_json=Path(cfg["stage2_input_json"]),
    stage2_output_json=Path(cfg["stage2_tokenized_json"]),
    stage1_max_len=128,
    stage2_max_len=256,
    overwrite=False,
)

print("Tokenizer:", result["tokenizer_id"])
print("Stage 1:", result["stage1_mode"], "->", cfg["stage1_tokenized_json"])
print("Stage 2:", result["stage2_mode"], "->", cfg["stage2_tokenized_json"])
print("Stage 2 input used:", result["stage2_input_used"])


## Stage 3: CLIP Feature Precompute

Inputs:
- Tokenized stage 1 data
- Raw images

Output:
- `.npy` CLIP embeddings in `cfg["clip_feature_dir"]`


### Checklist - Before Stage 3
- [ ] Stage 1 tokenized JSON exists
- [ ] `image_root` points to extracted `train2017/` images
- [ ] `clip_feature_dir` has enough storage for many `.npy` files
- [ ] Delete old feature files first if you want fresh regeneration


In [ ]:
from pathlib import Path
from src.vision_features.feature_pipeline import run_stage1_clip_precompute

result = run_stage1_clip_precompute(
    tokenized_json=Path(cfg["stage1_tokenized_json"]),
    image_root=Path(cfg["image_root"]),
    output_dir=Path(cfg["clip_feature_dir"]),
    clip_model_dir=cfg.get("clip_model_dir"),
    download_if_missing=True,
    overwrite=False,
)

print("Stage 3 mode:", result["mode"])
print("Feature dir:", result["output_dir"])
print("Feature files:", result["num_feature_files"])


## Stage 4: Manifest Build

Build stage-specific manifests that join text tokens with vision feature paths.


### Checklist - Before Stage 4
- [ ] Stage 1 tokenized JSON exists (required)
- [ ] Stage 2 tokenized JSON exists (optional; only if building Stage 2 manifest now)
- [ ] `clip_feature_dir` exists and contains `.npy` feature files
- [ ] If manifest already exists, keep skip behavior or set `overwrite=True` intentionally
- [ ] If CLIP features are partial/incomplete, finish Stage 3 before relying on manifests for training


In [ ]:
from pathlib import Path
from src.training_manifests.manifest_pipeline import run_manifest_pipeline

result = run_manifest_pipeline(
    stage1_tokenized_json=Path(cfg["stage1_tokenized_json"]),
    stage2_tokenized_json=Path(cfg["stage2_tokenized_json"]),
    clip_feature_dir=Path(cfg["clip_feature_dir"]),
    stage1_manifest_json=Path(cfg["stage1_manifest_json"]),
    stage2_manifest_json=Path(cfg["stage2_manifest_json"]),
    overwrite=False,
)

print("Stage 4 results:")
print("  Stage 1:", result["stage1_mode"], "rows=", result["stage1_rows"], "->", cfg["stage1_manifest_json"])
print("  Stage 2:", result["stage2_mode"], "rows=", result["stage2_rows"], "->", cfg["stage2_manifest_json"])


## Stage 5: Stage 1 Training (Projector Alignment)

This stage trains projector parameters while keeping base LLM frozen.


## Stage 4.5: LLaMA Model Bootstrap

Load the base LLaMA weights/tokenizer into this notebook session so Stage 5 can train.

### Checklist - Before Stage 4.5
- [ ] HF auth works (`hf auth whoami`)
- [ ] Access approved for `meta-llama/Llama-3.2-1B-Instruct`
- [ ] Enough storage for model files in `data/models/`


In [ ]:
import os
from pathlib import Path
from src.model_internals.loader_pipeline import ensure_llama_artifacts, load_llama_model_and_tokenizer

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
repo_id = cfg.get("tokenizer_id", "meta-llama/Llama-3.2-1B-Instruct")
llama_local_dir = Path(cfg.get("llama_local_dir", str(PROJECT_ROOT / "data" / "models" / "Llama-3.2-1B-Instruct")))

artifacts_result = ensure_llama_artifacts(
    repo_id=repo_id,
    local_dir=llama_local_dir,
    force_download=False,
)

load_result = load_llama_model_and_tokenizer(
    local_dir=llama_local_dir,
    dtype="bfloat16",
    use_mesh=True,
)

llama_model = load_result["llama_model"]
tokenizer = load_result["tokenizer"]

print("Model artifacts:", artifacts_result["mode"], "->", artifacts_result["local_dir"])
print("llama_model initialized. devices=", load_result["num_devices"], "mesh=", load_result["mesh_enabled"])


### Checklist - Before Stage 5
- [ ] Stage 1 manifest exists
- [ ] Stage 4.5 model bootstrap completed (`llama_model` is loaded)
- [ ] Output checkpoint path is chosen and not accidentally overwritten
- [ ] Start with a short smoke run (small batch/epochs)


In [ ]:
from pathlib import Path
from src.training.train_pipeline import run_stage1_training_pipeline

llama_model = globals().get("llama_model", None)
if llama_model is None:
    print("Stage 5 prerequisite missing: llama_model is not initialized in this notebook session.")
    print("Run Stage 4.5 first, then rerun this cell.")
else:
    result = run_stage1_training_pipeline(
        manifest_json=Path(cfg["stage1_manifest_json"]),
        stage1_projector_state_path=Path(cfg["stage1_projector_state"]),
        llama_model=llama_model,
        learning_rate=1e-4,
        num_epochs=1,
        batch_size=2,
        log_every=10,
        overwrite=False,
        seed=0,
    )
    print("Stage 5 mode:", result["mode"])
    print("Projector state:", result["projector_state_path"])


## Stage 6: Stage 2 Training (Multimodal Finetuning)

This stage updates projector + LLM states jointly.


### Checklist - Before Stage 6
- [ ] Stage 5 completed (`stage1_projector_state` exists)
- [ ] `llama_model` is initialized (run Stage 4.5)
- [ ] Stage 2 manifest exists, or create a smoke manifest
- [ ] Save paths for stage-2 outputs are confirmed


In [ ]:
from pathlib import Path
from src.training.train_pipeline import (
    build_smoke_manifest_from_existing_features,
    run_stage2_training_pipeline,
)

llama_model = globals().get("llama_model", None)
if llama_model is None:
    print("Stage 6 prerequisite missing: llama_model is not initialized in this notebook session.")
    print("Run Stage 4.5 first, then rerun this cell.")
else:
    smoke_mode = True  # keep True for quick server tests; set False for full run
    manifest_path = Path(cfg["stage2_manifest_json"])

    if smoke_mode:
        smoke_manifest = Path(cfg.get(
            "stage2_manifest_smoke_json",
            str(PROJECT_ROOT / "data" / "processed" / "stage2_finetuning" / "stage2_manifest_smoke.json"),
        ))
        smoke_result = build_smoke_manifest_from_existing_features(
            source_manifest_json=manifest_path,
            output_manifest_json=smoke_manifest,
            max_rows=256,
        )
        manifest_path = smoke_manifest
        print("Smoke manifest:", smoke_result["kept_rows"], "rows ->", smoke_result["output_manifest_json"])

    result = run_stage2_training_pipeline(
        manifest_json=manifest_path,
        stage1_projector_state_path=Path(cfg["stage1_projector_state"]),
        stage2_projector_state_path=Path(cfg["stage2_projector_state"]),
        stage2_llama_state_path=Path(cfg["stage2_llama_state"]),
        llama_model=llama_model,
        learning_rate=1e-5,
        num_epochs=1,
        batch_size=2,
        log_every=10,
        overwrite=True,
    )

    print("Stage 6 mode:", result["mode"])
    print("Stage 2 projector:", result["stage2_projector_state_path"])
    print("Stage 2 llama:", result["stage2_llama_state_path"])


## Stage 7: Artifacts + Quick Validation

Minimal checks to verify pipeline outputs exist before evaluation.


### Checklist - Before Stage 7
- [ ] Expected output files are defined in config
- [ ] You know which files are from Stage 1 vs Stage 2
- [ ] Missing files are investigated before moving to evaluation


In [ ]:
from pathlib import Path

checks = [
    cfg["stage1_chat_json"],
    cfg["stage1_tokenized_json"],
    cfg["stage1_manifest_json"],
    cfg["stage2_tokenized_json"],
    cfg["stage2_manifest_json"],
]

for p in checks:
    exists = Path(p).exists()
    print(f"{p} -> {'OK' if exists else 'MISSING'}")
